# 13 Pooling Metric Comparison

## Purpose

This notebook compares notebook-8 similarity outputs for the **same exact checkpoint** under three pooling rules:
- `cls`
- `mean`
- `max`

The goal is to answer the professor's question more directly. I do **not** just want to show whether max pooling shifts cosine scores upward. I want to check whether each pooling rule does a better job of keeping similar glycans near each other and dissimilar glycans farther apart.

This notebook stays centered on the matched global comparison. A single glycan such as `G74120DW` can still be inspected when follow-up is needed, but that is not the main structure of the analysis.


## Setup note

Same Colab pattern as the other report notebooks:
- code lives in GitHub
- notebook 8 outputs live in Drive
- this notebook pulls the repo and imports helpers from `src/`

This implementation treats the current Drive layout as the source of truth:
- `DRIVE_ROOT = /content/drive/MyDrive/ProjectRoot`
- notebook 8 outputs under `results/similarity_scaleup/`
- notebook 13 outputs under `results/similarity_model_comparison/`
- clean public exports under `results/public_reports/`

This notebook does not load model weights. It reads the saved notebook-8 CSVs and confirms that the `cls`, `mean`, and `max` runs all point back to the same underlying checkpoint folder before it compares them.


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Active repo directory: {REPO_DIR}')


## User settings

This is the main cell to edit when I want to rerun the workflow for a different tokenizer or checkpoint family.

The crucial matching rule is: **hold the checkpoint fixed and only change the pooling rule**.

So the three notebook-8 folders that feed this notebook must all agree on:
- tokenizer family
- experiment / training run
- exact checkpoint folder
- selected query glycans
- corpus / test split

Only the pooling strategy should differ.


In [ ]:
# ==============================================================================
# 1. IMPORT HELPERS AND DEFINE THE MATCHED POOLING COMPARISON
# ==============================================================================
import importlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display

import src.similarity_pooling_comparison as similarity_pooling_comparison
importlib.reload(similarity_pooling_comparison)

from src.similarity import (
    build_pooling_metric_comparison,
    build_pooling_run_specs,
    build_public_export_dir,
    build_public_report_subdir,
    export_public_pooling_metric_comparison_html,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
SUPPORTED_TOKENIZER_FAMILIES = (
    'byte_bpe',
    'glyberta',
    'manual',
    'hybrid_char_bpe',
    'linkage_block',
    'donor_bound',
    'semi_atomic',
)

TOKENIZER_FAMILY = 'manual'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'

# Choose one exact checkpoint family at a time. The notebook will then look for
# exactly three matched notebook-8 outputs: cls, mean, and max.
CHECKPOINT_FAMILY = 'classification_mlm_init'
BASE_OUTPUT_RUN_LABEL = 'live_extended'

CLASSIFIER_MLM_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_mlm'
CLASSIFIER_RANDOM_RUN_LABEL = 'cls_lr2e-5_ep100_bs16_randominit'

CHECKPOINT_FAMILY_CONFIGS = {
    'pretrained_mlm': {
        'checkpoint_source': 'pretraining',
        'model_output_id': 'pretrained_mlm',
        'classifier_run_label': None,
        'display_label': 'Pretrained MLM',
    },
    'classification_mlm_init': {
        'checkpoint_source': 'classification',
        'model_output_id': CLASSIFIER_MLM_RUN_LABEL,
        'classifier_run_label': CLASSIFIER_MLM_RUN_LABEL,
        'display_label': 'Classifier, MLM init',
    },
    'classification_random_init': {
        'checkpoint_source': 'classification',
        'model_output_id': CLASSIFIER_RANDOM_RUN_LABEL,
        'classifier_run_label': CLASSIFIER_RANDOM_RUN_LABEL,
        'display_label': 'Classifier, random init',
    },
}

if TOKENIZER_FAMILY not in SUPPORTED_TOKENIZER_FAMILIES:
    raise ValueError(f'Unsupported tokenizer family: {TOKENIZER_FAMILY}')

if CHECKPOINT_FAMILY not in CHECKPOINT_FAMILY_CONFIGS:
    raise ValueError(f'Unsupported CHECKPOINT_FAMILY: {CHECKPOINT_FAMILY}')

checkpoint_family_config = CHECKPOINT_FAMILY_CONFIGS[CHECKPOINT_FAMILY]
SIMILARITY_SCALEUP_ROOT = DRIVE_ROOT / 'results' / 'similarity_scaleup'

RUN_SPECS = build_pooling_run_specs(
    scaleup_results_root=SIMILARITY_SCALEUP_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    checkpoint_source=checkpoint_family_config['checkpoint_source'],
    model_output_id=checkpoint_family_config['model_output_id'],
    base_output_run_label=BASE_OUTPUT_RUN_LABEL,
    classifier_run_label=checkpoint_family_config['classifier_run_label'],
)

for spec in RUN_SPECS:
    spec['model_label'] = (
        f"{checkpoint_family_config['display_label']} | {spec['pooling_strategy'].upper()}"
    )

COMPARISON_RUN_LABEL = f'{CHECKPOINT_FAMILY}__{BASE_OUTPUT_RUN_LABEL}__cls_mean_max'
OUTPUT_DIR = (
    DRIVE_ROOT
    / 'results'
    / 'similarity_model_comparison'
    / TOKENIZER_FAMILY
    / EXPERIMENT_NAME
    / 'pooling_metric_comparison'
    / COMPARISON_RUN_LABEL
)

TOP_K_NEIGHBORS = 25
INSPECT_QUERY_ACCESSIONS = []
INSPECT_TOP_N = 12
SCATTER_SAMPLE_SIZE = 20000
HTML_REPORT_TITLE = (
    f'{TOKENIZER_FAMILY} pooling metric comparison | {checkpoint_family_config["display_label"]}'
)
EMBED_HTML_IMAGES = True

PUBLIC_EXPORT_ENABLED = True
PUBLIC_EXPORT_PARENT_SUBDIR = 'results/public_reports'
PUBLIC_GITHUB_OWNER = 'hb791-dev'
PUBLIC_GITHUB_REPO = 'glycan-roberta'
PUBLIC_GITHUB_REF = 'main'
PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH = True
PUBLIC_REPORT_NOTEBOOK_STEM = '13_pooling_metric_comparison'
PUBLIC_EXPORT_PATH_PARTS = [
    TOKENIZER_FAMILY,
    EXPERIMENT_NAME,
    CHECKPOINT_FAMILY,
    COMPARISON_RUN_LABEL,
]
PUBLIC_EXPORT_PARENT_DIR = DRIVE_ROOT / PUBLIC_EXPORT_PARENT_SUBDIR
PUBLIC_EXPORT_REPO_SUBDIR = build_public_report_subdir(
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
PUBLIC_EXPORT_DIR = build_public_export_dir(
    PUBLIC_EXPORT_PARENT_DIR,
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)

def stringify_config_paths(config_dict):
    return {
        key: str(value) if isinstance(value, Path) else value
        for key, value in config_dict.items()
    }

comparison_config = {
    'drive_root': str(DRIVE_ROOT),
    'tokenizer_family': TOKENIZER_FAMILY,
    'experiment_name': EXPERIMENT_NAME,
    'checkpoint_family': CHECKPOINT_FAMILY,
    'checkpoint_display_label': checkpoint_family_config['display_label'],
    'base_output_run_label': BASE_OUTPUT_RUN_LABEL,
    'comparison_run_label': COMPARISON_RUN_LABEL,
    'output_dir': str(OUTPUT_DIR),
    'top_k_neighbors': TOP_K_NEIGHBORS,
    'inspect_query_accessions': INSPECT_QUERY_ACCESSIONS,
    'inspect_top_n': INSPECT_TOP_N,
    'scatter_sample_size': SCATTER_SAMPLE_SIZE,
    'html_report_title': HTML_REPORT_TITLE,
    'embed_html_images': EMBED_HTML_IMAGES,
    'run_specs': [
        stringify_config_paths(spec)
        for spec in RUN_SPECS
    ],
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'pooling_metric_comparison_config.json').write_text(
    json.dumps(comparison_config, indent=2),
    encoding='utf-8',
)

print(f'Comparison output dir: {OUTPUT_DIR}')
print(f'Checkpoint family: {CHECKPOINT_FAMILY} -> {checkpoint_family_config["display_label"]}')
print(f'Base notebook-8 run label stem: {BASE_OUTPUT_RUN_LABEL}')
print('Matched notebook-8 run folders:')
for spec in RUN_SPECS:
    run_dir = Path(spec['run_dir'])
    print(
        f"- {spec['pooling_strategy'].upper()}: {run_dir} | exists={run_dir.exists()} | "
        f"ranked_csv_exists={(run_dir / 'specific_vs_all_ranked.csv').exists()}"
    )
print(f'Repo public_reports target: {PUBLIC_EXPORT_REPO_SUBDIR}')


## Build the matched comparison

This cell is the main workflow step.

It reads the three notebook-8 output folders, verifies that they are matched on the same underlying checkpoint, merges the saved `specific_vs_all_ranked.csv` tables on:
- `query_accession`
- `corpus_accession`
- `query_sequence`
- `corpus_sequence`

Then it saves the merged CSVs, summary tables, the 3x3 plot matrix, and a browser-friendly HTML report.


In [ ]:
# ==============================================================================
# 2. BUILD THE MATCHED POOLING COMPARISON
# ==============================================================================
comparison_outputs = build_pooling_metric_comparison(
    run_specs=RUN_SPECS,
    output_dir=OUTPUT_DIR,
    report_title=HTML_REPORT_TITLE,
    top_k_neighbors=TOP_K_NEIGHBORS,
    inspect_query_accessions=INSPECT_QUERY_ACCESSIONS,
    inspect_top_n=INSPECT_TOP_N,
    scatter_sample_size=SCATTER_SAMPLE_SIZE,
    embed_html_images=EMBED_HTML_IMAGES,
)

comparison_tables = comparison_outputs['tables']
matched_outputs = comparison_outputs['matched_outputs']
shared_model_dir = matched_outputs['shared_model_dir']

print(f'Matched shared checkpoint folder: {shared_model_dir}')
print('')
print('Saved comparison tables:')
for name, path in comparison_outputs['table_paths'].items():
    print(f'- {name}: {path}')

print('Saved comparison plots:')
for name, path in comparison_outputs['plot_paths'].items():
    print(f'- {name}: {path}')

print(f'Manifest: {comparison_outputs["manifest_path"]}')
print(f'HTML report: {comparison_outputs["plot_paths"]["html_report_path"]}')

display(HTML(
    f'<p><a href="{comparison_outputs["plot_paths"]["html_report_path"]}" target="_blank">Open full HTML pooling-comparison report</a></p>'
))

public_export_artifacts = None

if PUBLIC_EXPORT_ENABLED:
    public_export_artifacts = export_public_pooling_metric_comparison_html(
        comparison_outputs=comparison_outputs,
        export_dir=PUBLIC_EXPORT_DIR,
        repo_public_subdir=PUBLIC_EXPORT_REPO_SUBDIR,
        repo_owner=PUBLIC_GITHUB_OWNER,
        repo_name=PUBLIC_GITHUB_REPO,
        repo_ref=PUBLIC_GITHUB_REF,
    )

    print(f'Public export Drive folder: {public_export_artifacts["public_export_dir"]}')
    print(f'Repo folder to copy into before push: {PUBLIC_EXPORT_REPO_SUBDIR}')
    print(f'Repo report path after push: {public_export_artifacts["repo_index_path"]}')
    print(f'GitHack URL after push: {public_export_artifacts["githack_url"]}')
    print('')

    print('=== Copied public files ===')
    display(public_export_artifacts['copied_files_df'])

    print('=== Dependency issues ===')
    if public_export_artifacts['dependency_issues_df'].empty:
        print('No missing local HTML dependencies were found in the public export.')
    else:
        display(public_export_artifacts['dependency_issues_df'])

    print('=== Sensitive-string scan ===')
    if public_export_artifacts['scan_results_df'].empty:
        print('No obvious personal paths or local-environment strings were found in the copied files.')
    else:
        display(public_export_artifacts['scan_results_df'])

    if public_export_artifacts['has_dependency_issues']:
        raise ValueError(
            'The public export still has missing local dependencies. Fix those before pushing.'
        )

    if public_export_artifacts['has_sensitive_matches'] and PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH:
        raise ValueError(
            'The public export still contains suspicious local strings. Review the scan table before pushing.'
        )
else:
    print('PUBLIC_EXPORT_ENABLED is False, so this notebook skipped the GitHub/GitHack export step.')


## Confirm the checkpoint match

This table is the first sanity check to read before interpreting any plots.

If the `model_dir`, query counts, or corpus counts differ across rows, then the comparison is not valid as a pooling-only comparison. The helper code above already enforces that match, but I still show it explicitly so the notebook output is presentation-ready.


In [ ]:
# ==============================================================================
# 3. REVIEW THE MATCHED RUN SUMMARY
# ==============================================================================
display(comparison_tables['matched_pooling_run_summary'])


## Review the global pooling behavior

The 3x3 matrix is the main visual answer to the professor's question.

How to read it:
- diagonal histograms show the score distribution for each pooling rule
- off-diagonal scatter plots compare the **same matched query-corpus pairs** between two pooling rules
- a mostly shifted but tight diagonal cloud suggests a location or scale change
- a broad cloud with noticeable spread away from the diagonal suggests the ranking or neighborhood structure is changing

Because the merged table comes from notebook 8 outputs, this is still a comparison of the same real held-out corpus and the same selected query glycans.


In [ ]:
# ==============================================================================
# 4. REVIEW GLOBAL SUMMARIES AND THE 3X3 MATRIX
# ==============================================================================
display(comparison_tables['pooling_similarity_summary'])
display(comparison_tables['pooling_pairwise_correlations'])
display(Image(filename=comparison_outputs['plot_paths']['pooling_matrix_plot']))


## Review matched neighborhoods

These tables keep the checkpoint fixed and compare whether the top neighbors stay stable across `cls`, `mean`, and `max`.

This gets closer to the scientific question than a simple score-shift plot because it asks whether each pooling rule preserves the same nearest-neighbor structure for each query glycan.


In [ ]:
# ==============================================================================
# 5. REVIEW MATCHED TOP-K NEIGHBOR OVERLAP
# ==============================================================================
display(comparison_tables['pooling_top_k_overlap_by_query'])
display(comparison_tables['pooling_top_k_overlap_summary'])
display(comparison_tables['pooling_top_k_three_way_overlap'])


## Inspect individual query glycans when needed

The notebook is not organized around a single glycan by default, but this section is here for follow-up examples such as `G74120DW`.

Leave `INSPECT_QUERY_ACCESSIONS = []` to keep the notebook broad, or set it to one or more query accessions when I want an ad hoc drill-down after the global comparison raises a question.


In [ ]:
# ==============================================================================
# 6. REVIEW PER-QUERY SUMMARIES AND THE INSPECTION TABLE
# ==============================================================================
display(comparison_tables['pooling_similarity_summary_by_query'])
display(comparison_tables['pooling_query_inspection'])


## Saved outputs

This notebook writes:
- merged matched score tables and summary CSVs into the notebook-13 output folder
- the 3x3 plot matrix as a PNG
- a self-contained HTML report for Drive review
- an optional clean public export folder that can be copied straight into `public_reports/13_pooling_metric_comparison/...`


In [ ]:
# ==============================================================================
# 7. PRINT THE SAVED OUTPUT PATHS
# ==============================================================================
print('Saved table outputs:')
for label, path in comparison_outputs['table_paths'].items():
    print(f'- {label}: {path}')

print('Saved plot outputs:')
for label, path in comparison_outputs['plot_paths'].items():
    print(f'- {label}: {path}')

print(f'Manifest: {comparison_outputs["manifest_path"]}')
